In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Sun Jun 21 14:54:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             44W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
  "transformers==4.53.3" \
  "peft==0.17.1" \
  "trl" \
  "accelerate" \
  "bitsandbytes" \
  "wandb"

In [4]:
import os
import math
import torch
import transformers
import peft

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)

from datetime import datetime
from transformers import (
    AutoModelForMaskedLM, 
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, Dataset

PyTorch version: 2.11.0+cu128
Transformers version: 4.53.3
PEFT version: 0.17.1


# Utilities

In [5]:
def load_hf_dataset(
    lang, # 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'xquad' | 'xcopa'
    split, size,
):
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
        }
        # TODO: Add configurations for 'xquad' and 'xcopa'
    }
    
    # Validate task input
    assert task in data_configs, f"Unsupported task: {task}. Supported tasks: {list(data_configs.keys())}"
    
    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']

    # Use streaming
    dataset_stream = load_dataset(data_id, data_dir=data_dir, split=split, streaming=True)

    # Slice the dataset to the specified size
    sliced_data = []
    for i, example in enumerate(dataset_stream):
        if i >= size:
            break
        sliced_data.append(example)

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

# Configurations

In [6]:
# Run configuration
SEED = 42
USERNAME = 'alxxtexxr'
LANG = 'en'  # 'en' | 'ja' | 'id'
TASK = 'wikipedia'  # 'wikipedia' | 'xquad' | 'xcopa'

# Data configuration
DATA_SPLIT = 'train'
DATA_SIZE = 1250
TEST_RATIO = 0.1

# Calculate train and test sizes based on the specified test ratio
# train_size = int(DATA_SIZE * (1 - TEST_RATIO))
# test_size = DATA_SIZE - train_size

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = 'all-linear'

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 20
WARMUP_STEPS = 50
LR = 2e-4
MLM_PROB = 0.15

In [7]:
# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist!"
            
else:
    # TODO: Handle the Hugging Face username properly
    model_name = MODEL_ID
    run_name = MODEL_ID.split('/')[-1]
    run_name = (
        f'{run_name.split("-v")[0] if "-v" in run_name else run_name}'
        f'-{TASK}-{LANG}-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    )
    hub_model_id = f'{USERNAME}/{run_name}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)

Resume from checkpoint: False
Model name: FacebookAI/xlm-roberta-base
Run name: xlm-roberta-base-wikipedia-en-LoRA-v260621145421
Hub model ID: alxxtexxr/xlm-roberta-base-wikipedia-en-LoRA-v260621145421


In [8]:
# Set environment variables for wandb logging
os.environ['WANDB_PROJECT'] = 'legamex'
os.environ['WANDB_NAME'] = run_name
# os.environ['WANDB_LOG_MODEL'] = 'checkpoint' # Control whether checkpoints get uploaded to wandb as artifacts

# Model

In [9]:
# Load the model and tokenizer
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)

# Set up LoRA configuration and apply it to the model
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='FEATURE_EXTRACTION', # 'CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("Model device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model tha

trainable params: 2,678,784 || all params: 280,973,970 || trainable%: 0.9534
Model device: cpu


# Data

In [10]:
# Load the dataset
dataset = load_hf_dataset(lang=LANG, task=TASK, split=DATA_SPLIT, size=DATA_SIZE)
print(dataset)

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1250
})


In [11]:
# Tokenize the dataset
def tokenize_fn(examples):
    outputs = tokenizer(
        examples['text'],
        max_length=512,
        truncation=True,
        padding='max_length',
    )
    outputs['labels'] = outputs['input_ids'].copy()
    return outputs
dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

# Perform train/val/test split
split1 = dataset.train_test_split(
    test_size=TEST_RATIO * 2,
    seed=SEED,
)

train_dataset = split1['train']
tmp_dataset = split1['test']

split2 = tmp_dataset.train_test_split(
    test_size=0.5,
    seed=SEED,
)

val_dataset = split2['train']
test_dataset = split2['test']

print("Train dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)
print()
print("Test dataset:")
print(test_dataset)

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Train dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})

Validation dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 125
})

Test dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 125
})


# Training

In [12]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROB,
)

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

training_args = TrainingArguments(
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=20,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=10,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=20,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better = False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=3,
            # early_stopping_threshold = 0.001,
        )
    ],
)
# Explicitly tell the trainer which column contains the labels
trainer.label_names = ['labels']

Calculated max steps: 1260


No label_names provided for model class `PeftModelForFeatureExtraction`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
20,1.541600,1.454164
40,1.571200,1.433448
60,1.488400,1.346364
80,1.510400,1.425135


: 